# mini-infer: correctness first, then GPU measurements

Choose a GPU runtime (a free T4 is sufficient), then run cells in order. Hardware is detected, never assumed. This notebook has not been executed on a T4 by the project author. The pinned environment needs Python 3.11 or 3.12; incompatible runtime versions fail before installation. Change `DEVICE` to `cpu` for an explicitly CPU-labeled fallback. Downloads are public and need no key.


In [ ]:
import os, subprocess, sys
from pathlib import Path
assert sys.version_info[:2] in ((3, 11), (3, 12)), "Use a Python 3.11/3.12 runtime compatible with torch==2.5.1"
REVISION = "codex/mini-infer-m8"  # Replace with a reviewed commit SHA or main after merge.
DEVICE = "cuda"  # Set to "cpu" only for the CPU fallback.
PROJECT = Path("/content/mini-infer")
URL = "https://github.com/harshith49/mini-infer.git"
if not PROJECT.exists():
    subprocess.run(["git", "clone", URL, str(PROJECT)], check=True)
assert subprocess.check_output(["git", "remote", "get-url", "origin"], cwd=PROJECT, text=True).strip() == URL
subprocess.run(["git", "fetch", "origin"], cwd=PROJECT, check=True)
subprocess.run(["git", "checkout", REVISION], cwd=PROJECT, check=True)
print(subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True))
ENV = dict(os.environ, HF_HOME=str(PROJECT / "model_cache"), OMP_NUM_THREADS="1", TOKENIZERS_PARALLELISM="false")
VENV = Path("/content/mini-infer-venv")
subprocess.run([sys.executable, "-m", "venv", str(VENV)], check=True)
PYTHON = str(VENV / "bin/python")
def run(*args):
    subprocess.run([PYTHON, *args], cwd=PROJECT, env=ENV, check=True)
run("-m", "pip", "install", "-r", "requirements.txt")
run("-m", "pip", "check")


In [ ]:
run("-c", "import platform, torch; print(platform.python_version(), torch.__version__); print('CUDA:', torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')")
assert DEVICE in ("cpu", "cuda")
if DEVICE == "cuda":
    run("-c", "import torch; assert torch.cuda.is_available(), 'Enable a GPU runtime, or explicitly select CPU fallback'")
# All commands are checked. A failed test prevents benchmarks in this cell.
run("-m", "pytest", "-q")
run("-m", "benchmarks.bench_stages", "--suite", "all", "--device", DEVICE, "--output", f"results/stages_{DEVICE}.csv")
run("-m", "benchmarks.bench_vs_hf", "--device", DEVICE, "--output", f"results/vs_hf_{DEVICE}.csv")
run("-m", "benchmarks.make_charts", "--input", f"results/stages_{DEVICE}.csv")


In [ ]:
from IPython.display import Image, display
for name in ("throughput", "latency", "memory"):
    display(Image(filename=str(PROJECT / "results" / f"stages_{name}.png")))
import shutil
archive = shutil.make_archive("/content/mini-infer-results", "zip", PROJECT / "results")
from google.colab import files
files.download(archive)


## Optional vLLM comparison

Run this in a **separate** Python 3.11 CUDA environment, following the [vLLM 0.8.5 installation guide](https://docs.vllm.ai/en/v0.8.5/getting_started/installation/gpu.html). Do not install vLLM over the pinned primary environment: it has its own Torch/Transformers dependencies. For that environment, install `vllm==0.8.5` and `httpx`, then run `python -m benchmarks.bench_vs_vllm` from this checkout. API/runtime failures are visible; absence of vLLM/CUDA produces an explicit skip and no CSV. This offline script observes whole-cohort return latency, so its p50/p95 are not per-request delivery measurements. Its worker memory is unmeasured. No vLLM GPU result is committed.

## Interpretation

Each primary row measures the same 216 useful output tokens, three repetitions after a separate warmup, fixed-budget greedy requests and one thread. Whole-request latency includes queue waiting. Static cohorts perform additional trimmed work. The final int8 stage includes paged KV and must pass the existing 5% perplexity-increase quality gate. CUDA memory is PyTorch allocated peak including the active model, not total process/driver memory. CPU peak is unmeasured. Runtime download/loading, conversion and quality scoring are outside timers. The HF comparison is sequential cached generation, a separate pairing from continuous serving. Preserve raw CSVs even when an optimization is slower.
